In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_CURRENT = "sentinel_dev.silver.silver_orders_current"

# DIM_CUSTOMER = "sentinel_dev.gold.dim_customer"
# DIM_PRODUCT = "sentinel_dev.gold.dim_product"
# FACT_ORDERS = "sentinel_dev.gold.fact_orders"
# DAILY_SALES = "sentinel_dev.gold.daily_sales"
CONTROL_TABLE = "sentinel_dev.monitoring.gold_watermark"

In [0]:
orders_df = spark.table(SILVER_CURRENT)

print(f"Current valid orders: {orders_df.count():,}")

orders_df.printSchema()

In [0]:
dim_product_df = (
    orders_df
        .select(
            "product_id",
            "product_name"
        )
        .filter(F.col("product_id").isNotNull())
        .dropDuplicates(["product_id"])

        .withColumn(
            "product_key",
            F.xxhash64("product_id")
        )

        .select(
            "product_key",
            "product_id",
            "product_name"
        )
)

In [0]:
display(dim_product_df.limit(20))

In [0]:
if not spark.catalog.tableExists(CONTROL_TABLE):
    (
        spark.createDataFrame(
            [("fact_orders", None)],
            "pipeline_name STRING, last_processed_at TIMESTAMP"
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(CONTROL_TABLE)
    )

print("Gold watermark table ready.")

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

print(f"Last processed at: {last_processed_at}")

In [0]:
silver_current_df = spark.table(
    "sentinel_dev.silver.silver_orders_current"
)

In [0]:
if last_processed_at is None:
    changed_orders_df = silver_current_df
else:
    changed_orders_df = (
        silver_current_df
            .filter(
                F.col("ingested_at") > F.lit(last_processed_at)
            )
    )

print(
    f"Changed Silver orders to process: "
    f"{changed_orders_df.count():,}"
)

In [0]:
dim_customer_df = spark.table(
    "sentinel_dev.gold.dim_customer"
)

dim_product_df = spark.table(
    "sentinel_dev.gold.dim_product"
)


In [0]:
if last_processed_at is None:
    changed_orders_df = silver_current_df
else:
    changed_orders_df = (
        silver_current_df
            .filter(
                F.col("ingested_at") > F.lit(last_processed_at)
            )
    )

print(
    f"Changed Silver orders to process: "
    f"{changed_orders_df.count():,}"
)

In [0]:
print(
    f"Changed Silver orders to process: "
    f"{changed_orders_df.count():,}"
)

display(
    changed_orders_df
        .select(
            "order_id",
            "order_status",
            "order_timestamp_clean",
            "ingested_at"
        )
)

In [0]:
gold_changes_df = (
    changed_orders_df.alias("o")

        .join(
            dim_customer_df.alias("c"),
            F.col("o.customer_id") == F.col("c.customer_id"),
            "left"
        )

        .join(
            dim_product_df.alias("p"),
            F.col("o.product_id") == F.col("p.product_id"),
            "left"
        )

        .select(
            F.col("o.order_id"),

            F.col("c.customer_key"),
            F.col("p.product_key"),

            F.to_date(
                F.col("o.order_timestamp_clean")
            ).alias("order_date"),

            F.col("o.order_timestamp_clean")
                .alias("order_timestamp"),

            F.col("o.quantity"),
            F.col("o.unit_price"),
            F.col("o.total_amount"),

            F.col("o.payment_method"),
            F.col("o.order_status"),
            F.col("o.source_system"),

            F.col("o.ingested_at")
        )
)

In [0]:
from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(
    spark,
    "sentinel_dev.gold.fact_orders"
)

(
    gold_delta.alias("target")

    .merge(
        gold_changes_df.alias("source"),
        "target.order_id = source.order_id"
    )

    .whenMatchedUpdateAll()

    .whenNotMatchedInsertAll()

    .execute()
)

print("Incremental Gold MERGE completed.")

In [0]:
new_watermark = (
    changed_orders_df
        .agg(
            F.max("ingested_at").alias("max_ingested_at")
        )
        .first()["max_ingested_at"]
)

In [0]:
if new_watermark is not None:

    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET last_processed_at = TIMESTAMP '{new_watermark}'
        WHERE pipeline_name = 'fact_orders'
    """)

    print(
        f"Watermark advanced to: {new_watermark}"
    )

In [0]:
display(
    spark.table(CONTROL_TABLE)
)

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

changed_after_run = (
    silver_current_df
        .filter(
            F.col("ingested_at") > F.lit(last_processed_at)
        )
        .count()
)

print(
    f"Rows requiring processing after successful run: "
    f"{changed_after_run}"
)

In [0]:
watermark_row = (
    spark.table(CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_at")
        .first()
)

last_processed_at = watermark_row["last_processed_at"]

changed_after_run = (
    silver_current_df
        .filter(
            F.col("ingested_at") > F.lit(last_processed_at)
        )
        .count()
)

print(
    f"Rows requiring processing after successful run: "
    f"{changed_after_run}"
)

In [0]:
display(
    spark.table("sentinel_dev.silver.silver_orders_current")
        .filter(
            F.col("order_id").isin(
                "DUP-ORDER-001"
            )
        )
        .select(
            "order_id",
            "order_status",
            "order_timestamp_clean",
            "ingested_at"
        )
)

In [0]:
%sql
SELECT
    order_id,
    order_status,
    order_timestamp,
    total_amount
FROM sentinel_dev.gold.fact_orders
WHERE order_id = 'INC-ORDER-002';

In [0]:
silver_table = "sentinel_dev.silver.silver_orders_current"

history_df = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
)

display(
    history_df.select(
        "version",
        "timestamp",
        "operation"
    ).orderBy(F.desc("version"))
)

In [0]:
silver_table = "sentinel_dev.silver.silver_orders_current"

history_df = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
)

display(
    history_df.select(
        "version",
        "timestamp",
        "operation"
    ).orderBy(F.desc("version"))
)

In [0]:
CDF_CONTROL_TABLE = "sentinel_dev.monitoring.gold_cdf_checkpoint"

if not spark.catalog.tableExists(CDF_CONTROL_TABLE):
    spark.createDataFrame(
        [("fact_orders", None)],
        "pipeline_name STRING, last_processed_version LONG"
    ).write.format("delta").mode("overwrite").saveAsTable(
        CDF_CONTROL_TABLE
    )

display(spark.table(CDF_CONTROL_TABLE))

In [0]:
checkpoint = (
    spark.table(CDF_CONTROL_TABLE)
        .filter(F.col("pipeline_name") == "fact_orders")
        .select("last_processed_version")
        .first()
)

last_processed_version = checkpoint["last_processed_version"]

print(
    f"Last processed Delta version: "
    f"{last_processed_version}"
)

In [0]:
current_version = (
    spark.sql(f"DESCRIBE HISTORY {silver_table}")
        .agg(F.max("version").alias("version"))
        .first()["version"]
)

print(f"Current Silver version: {current_version}")

In [0]:
if last_processed_version is None:
    spark.sql(f"""
        UPDATE {CDF_CONTROL_TABLE}
        SET last_processed_version = {current_version}
        WHERE pipeline_name = 'fact_orders'
    """)

    last_processed_version = current_version

    print(
        f"CDF checkpoint bootstrapped at version "
        f"{current_version}"
    )

In [0]:
current_version = (
    spark.sql(
        "DESCRIBE HISTORY sentinel_dev.silver.silver_orders_current"
    )
    .agg(F.max("version").alias("version"))
    .first()["version"]
)

print(f"Current Silver version: {current_version}")

In [0]:
starting_version = last_processed_version + 1

cdf_df = (
    spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", starting_version)
        .table("sentinel_dev.silver.silver_orders_current")
)

display(
    cdf_df.select(
        "order_id",
        "order_status",
        "order_timestamp_clean",
        "_change_type",
        "_commit_version",
        "_commit_timestamp"
    )
)

In [0]:
cdf_changes_df = (
    cdf_df
        .filter(
            F.col("_change_type").isin(
                "insert",
                "update_postimage"
            )
        )
)

display(
    cdf_changes_df.select(
        "order_id",
        "order_status",
        "_change_type",
        "_commit_version"
    )
)

In [0]:
dim_customer_df = spark.table(
    "sentinel_dev.gold.dim_customer"
)

dim_product_df = spark.table(
    "sentinel_dev.gold.dim_product"
)

gold_cdf_changes_df = (
    cdf_changes_df.alias("o")

    .join(
        dim_customer_df.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )

    .join(
        dim_product_df.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )

    .select(
        F.col("o.order_id"),
        F.col("c.customer_key"),
        F.col("p.product_key"),

        F.to_date(
            F.col("o.order_timestamp_clean")
        ).alias("order_date"),

        F.col("o.order_timestamp_clean")
            .alias("order_timestamp"),

        F.col("o.quantity"),
        F.col("o.unit_price"),
        F.col("o.total_amount"),

        F.col("o.payment_method"),
        F.col("o.order_status"),
        F.col("o.source_system"),

        F.col("o.ingested_at")
    )
)

In [0]:
from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(
    spark,
    "sentinel_dev.gold.fact_orders"
)

(
    gold_delta.alias("target")
        .merge(
            gold_cdf_changes_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
)

print("CDF incremental MERGE completed.")

In [0]:
%sql
SELECT
    order_id,
    order_status,
    total_amount
FROM sentinel_dev.gold.fact_orders
WHERE order_id = 'CDF-ORDER-001';

In [0]:
max_processed_version = (
    cdf_changes_df
        .agg(
            F.max("_commit_version").alias("max_version")
        )
        .first()["max_version"]
)

if max_processed_version is not None:
    spark.sql(f"""
        UPDATE sentinel_dev.monitoring.gold_cdf_checkpoint
        SET last_processed_version = {max_processed_version}
        WHERE pipeline_name = 'fact_orders'
    """)

    print(
        f"CDF checkpoint advanced to version "
        f"{max_processed_version}"
    )